### 01 – EDA
#### HumanForYou – Attrition ML

Objectif : réaliser une exploration initiale des données RH avant le prétraitement.

- **Entrées** : fichiers CSV dans `data/raw/`
- **Sortie** : `data/processed/eda_summary.csv`
- **But** : contrôler la qualité des données, les valeurs manquantes et la cohérence des identifiants


##### SECTION – IMPORT

Cette cellule importe les bibliothèques nécessaires à l'analyse exploratoire et configure les chemins de données.

In [1]:
# Imports nécessaires pour l'analyse exploratoire et la gestion des chemins.
import pandas as pd
import numpy as np
import os

##### SECTION – LOAD DATA

Cette étape charge les fichiers bruts RH depuis `data/raw/` et prépare les DataFrames utilisés dans la suite.

In [2]:
# Chargement centralisé des jeux de données bruts depuis `data/raw`.
RAW_DIR = os.path.join('..', 'data', 'raw')
PROCESSED_DIR = os.path.join('..', 'data', 'processed')

general = pd.read_csv(os.path.join(RAW_DIR, 'general_data.csv'))
employee_survey = pd.read_csv(os.path.join(RAW_DIR, 'employee_survey_data.csv'))
manager_survey = pd.read_csv(os.path.join(RAW_DIR, 'manager_survey_data.csv'))
in_time = pd.read_csv(os.path.join(RAW_DIR, 'in_out_time', 'in_time.csv'))
out_time = pd.read_csv(os.path.join(RAW_DIR, 'in_out_time', 'out_time.csv'))

# Renommer la première colonne anonyme en EmployeeID
in_time.rename(columns={in_time.columns[0]: 'EmployeeID'}, inplace=True)
out_time.rename(columns={out_time.columns[0]: 'EmployeeID'}, inplace=True)

datasets = {
    'general': general,
    'employee_survey': employee_survey,
    'manager_survey': manager_survey,
    'in_time': in_time,
    'out_time': out_time
}

print('Chargement OK')
for name, df in datasets.items():
    print(f'  {name}: {df.shape}')


Chargement OK
  general: (4410, 24)
  employee_survey: (4410, 4)
  manager_survey: (4410, 3)
  in_time: (4410, 262)
  out_time: (4410, 262)


##### SECTION – BASIC CHECK

Ici, on contrôle rapidement la structure de chaque table (dimensions, types et aperçu des données).


In [3]:
# Vérification de base de chaque table (dimensions, aperçu, types).
for name, df in datasets.items():
    print(f'\n{"="*60}')
    print(f'  {name.upper()} — shape: {df.shape}')
    print(f'{"="*60}')
    display(df.head())
    print()
    df.info()
    print()



  GENERAL — shape: (4410, 24)


,Age,Attrition,BusinessTravel,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeID,Gender,...,NumCompaniesWorked,Over18,PercentSalaryHike,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,YearsAtCompany,YearsSinceLastPromotion,YearsWithCurrManager
0,51,No,Travel_Rarely,Sales,6,2,Life Sciences,1,1,Female,...,1.0,Y,11,8,0,1.0,6,1,0,0
1,31,Yes,Travel_Frequently,Research & Development,10,1,Life Sciences,1,2,Female,...,0.0,Y,23,8,1,6.0,3,5,1,4
2,32,No,Travel_Frequently,Research & Development,17,4,Other,1,3,Male,...,1.0,Y,15,8,3,5.0,2,5,0,3
3,38,No,Non-Travel,Research & Development,2,5,Life Sciences,1,4,Male,...,3.0,Y,11,8,3,13.0,5,8,7,5
4,32,No,Travel_Rarely,Research & Development,10,1,Medical,1,5,Male,...,4.0,Y,12,8,2,9.0,2,6,0,4



<class 'pandas.DataFrame'>
RangeIndex: 4410 entries, 0 to 4409
Data columns (total 24 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Age                      4410 non-null   int64  
 1   Attrition                4410 non-null   str    
 2   BusinessTravel           4410 non-null   str    
 3   Department               4410 non-null   str    
 4   DistanceFromHome         4410 non-null   int64  
 5   Education                4410 non-null   int64  
 6   EducationField           4410 non-null   str    
 7   EmployeeCount            4410 non-null   int64  
 8   EmployeeID               4410 non-null   int64  
 9   Gender                   4410 non-null   str    
 10  JobLevel                 4410 non-null   int64  
 11  JobRole                  4410 non-null   str    
 12  MaritalStatus            4410 non-null   str    
 13  MonthlyIncome            4410 non-null   int64  
 14  NumCompaniesWorked       4391 non-

,EmployeeID,EnvironmentSatisfaction,JobSatisfaction,WorkLifeBalance
0,1,3.0,4.0,2.0
1,2,3.0,2.0,4.0
2,3,2.0,2.0,1.0
3,4,4.0,4.0,3.0
4,5,4.0,1.0,3.0



<class 'pandas.DataFrame'>
RangeIndex: 4410 entries, 0 to 4409
Data columns (total 4 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   EmployeeID               4410 non-null   int64  
 1   EnvironmentSatisfaction  4385 non-null   float64
 2   JobSatisfaction          4390 non-null   float64
 3   WorkLifeBalance          4372 non-null   float64
dtypes: float64(3), int64(1)
memory usage: 137.9 KB


  MANAGER_SURVEY — shape: (4410, 3)


...


<class 'pandas.DataFrame'>
RangeIndex: 4410 entries, 0 to 4409
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   EmployeeID         4410 non-null   int64
 1   JobInvolvement     4410 non-null   int64
 2   PerformanceRating  4410 non-null   int64
dtypes: int64(3)
memory usage: 103.5 KB


  IN_TIME — shape: (4410, 262)


,EmployeeID,2015-01-01,2015-01-02,2015-01-05,2015-01-06,2015-01-07,2015-01-08,2015-01-09,2015-01-12,2015-01-13,...,2015-12-18,2015-12-21,2015-12-22,2015-12-23,2015-12-24,2015-12-25,2015-12-28,2015-12-29,2015-12-30,2015-12-31
0,1,NaN,2015-01-02 09:43:45,2015-01-05 10:08:48,2015-01-06 09:54:26,2015-01-07 09:34:31,2015-01-08 09:51:09,2015-01-09 10:09:25,2015-01-12 09:42:53,2015-01-13 10:13:06,...,NaN,2015-12-21 09:55:29,2015-12-22 10:04:06,2015-12-23 10:14:27,2015-12-24 10:11:35,NaN,2015-12-28 10:13:41,2015-12-29 10:03:36,2015-12-30 09:54:12,2015-12-31 10:12:44
1,2,NaN,2015-01-02 10:15:44,2015-01-05 10:21:05,NaN,2015-01-07 09:45:17,2015-01-08 10:09:04,2015-01-09 09:43:26,2015-01-12 10:00:07,2015-01-13 10:43:29,...,2015-12-18 10:37:17,2015-12-21 09:49:02,2015-12-22 10:33:51,2015-12-23 10:12:10,NaN,NaN,2015-12-28 09:31:45,2015-12-29 09:55:49,2015-12-30 10:32:25,2015-12-31 09:27:20
2,3,NaN,2015-01-02 10:17:41,2015-01-05 09:50:50,2015-01-06 10:14:13,2015-01-07 09:47:27,2015-01-08 10:03:40,2015-01-09 10:05:49,2015-01-12 10:03:47,2015-01-13 10:21:26,...,2015-12-18 10:15:14,2015-12-21 10:10:28,2015-12-22 09:44:44,2015-12-23 10:15:54,2015-12-24 10:07:26,NaN,2015-12-28 09:42:05,2015-12-29 09:43:36,2015-12-30 09:34:05,2015-12-31 10:28:39
3,4,NaN,2015-01-02 10:05:06,2015-01-05 09:56:32,2015-01-06 10:11:07,2015-01-07 09:37:30,2015-01-08 10:02:08,2015-01-09 10:08:12,2015-01-12 10:13:42,2015-01-13 09:53:22,...,2015-12-18 10:17:38,2015-12-21 09:58:21,2015-12-22 10:04:25,2015-12-23 10:11:46,2015-12-24 09:43:15,NaN,2015-12-28 09:52:44,2015-12-29 09:33:16,2015-12-30 10:18:12,2015-12-31 10:01:15
4,5,NaN,2015-01-02 10:28:17,2015-01-05 09:49:58,2015-01-06 09:45:28,2015-01-07 09:49:37,2015-01-08 10:19:44,2015-01-09 10:00:50,2015-01-12 10:29:27,2015-01-13 09:59:32,...,2015-12-18 09:58:35,2015-12-21 10:03:41,2015-12-22 10:10:30,2015-12-23 10:13:36,2015-12-24 09:44:24,NaN,2015-12-28 10:05:15,2015-12-29 10:30:53,2015-12-30 09:18:21,2015-12-31 09:41:09



<class 'pandas.DataFrame'>
RangeIndex: 4410 entries, 0 to 4409
Columns: 262 entries, EmployeeID to 2015-12-31
dtypes: float64(12), int64(1), str(249)
memory usage: 8.8 MB


  OUT_TIME — shape: (4410, 262)


,EmployeeID,2015-01-01,2015-01-02,2015-01-05,2015-01-06,2015-01-07,2015-01-08,2015-01-09,2015-01-12,2015-01-13,...,2015-12-18,2015-12-21,2015-12-22,2015-12-23,2015-12-24,2015-12-25,2015-12-28,2015-12-29,2015-12-30,2015-12-31
0,1,NaN,2015-01-02 16:56:15,2015-01-05 17:20:11,2015-01-06 17:19:05,2015-01-07 16:34:55,2015-01-08 17:08:32,2015-01-09 17:38:29,2015-01-12 16:58:39,2015-01-13 18:02:58,...,NaN,2015-12-21 17:15:50,2015-12-22 17:27:51,2015-12-23 16:44:44,2015-12-24 17:47:22,NaN,2015-12-28 18:00:07,2015-12-29 17:22:30,2015-12-30 17:40:56,2015-12-31 17:17:33
1,2,NaN,2015-01-02 18:22:17,2015-01-05 17:48:22,NaN,2015-01-07 17:09:06,2015-01-08 17:34:04,2015-01-09 16:52:29,2015-01-12 17:36:48,2015-01-13 18:00:13,...,2015-12-18 18:31:28,2015-12-21 17:34:16,2015-12-22 18:16:35,2015-12-23 17:38:18,NaN,NaN,2015-12-28 17:08:38,2015-12-29 17:54:46,2015-12-30 18:31:35,2015-12-31 17:40:58
2,3,NaN,2015-01-02 16:59:14,2015-01-05 17:06:46,2015-01-06 16:38:32,2015-01-07 16:33:21,2015-01-08 17:24:22,2015-01-09 16:57:30,2015-01-12 17:28:54,2015-01-13 17:21:25,...,2015-12-18 17:02:23,2015-12-21 17:20:17,2015-12-22 16:32:50,2015-12-23 16:59:43,2015-12-24 16:58:25,NaN,2015-12-28 16:43:31,2015-12-29 17:09:56,2015-12-30 17:06:25,2015-12-31 17:15:50
3,4,NaN,2015-01-02 17:25:24,2015-01-05 17:14:03,2015-01-06 17:07:42,2015-01-07 16:32:40,2015-01-08 16:53:11,2015-01-09 17:19:47,2015-01-12 17:13:37,2015-01-13 17:11:45,...,2015-12-18 17:55:23,2015-12-21 16:49:09,2015-12-22 17:24:00,2015-12-23 17:36:35,2015-12-24 16:48:21,NaN,2015-12-28 17:19:34,2015-12-29 16:58:16,2015-12-30 17:40:11,2015-12-31 17:09:14
4,5,NaN,2015-01-02 18:31:37,2015-01-05 17:49:15,2015-01-06 17:26:25,2015-01-07 17:37:59,2015-01-08 17:59:28,2015-01-09 17:44:08,2015-01-12 18:51:21,2015-01-13 18:14:58,...,2015-12-18 17:52:48,2015-12-21 17:43:35,2015-12-22 18:07:57,2015-12-23 18:00:49,2015-12-24 17:59:22,NaN,2015-12-28 17:44:59,2015-12-29 18:47:00,2015-12-30 17:15:33,2015-12-31 17:42:14



<class 'pandas.DataFrame'>
RangeIndex: 4410 entries, 0 to 4409
Columns: 262 entries, EmployeeID to 2015-12-31
dtypes: float64(12), int64(1), str(249)
memory usage: 8.8 MB



##### SECTION – MISSING VALUES

Cette section mesure les valeurs manquantes pour identifier les colonnes à traiter ensuite.

In [4]:
# Vérification de base de chaque table (dimensions, aperçu, types).
for name, df in datasets.items():
    missing = df.isnull().sum()
    missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
    missing_df = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
    missing_df = missing_df[missing_df['missing_count'] > 0].sort_values('missing_count', ascending=False)
    
    print(f'\n--- {name.upper()} ---')
    if missing_df.empty:
        print('Aucune valeur manquante')
    else:
        display(missing_df)
    print(f'Total NaN: {df.isnull().sum().sum()} / {df.size} ({(df.isnull().sum().sum()/df.size*100):.2f}%)')



--- GENERAL ---


,missing_count,missing_pct
NumCompaniesWorked,19,0.43
TotalWorkingYears,9,0.20


Total NaN: 28 / 105840 (0.03%)

--- EMPLOYEE_SURVEY ---


,missing_count,missing_pct
WorkLifeBalance,38,0.86
EnvironmentSatisfaction,25,0.57
JobSatisfaction,20,0.45


Total NaN: 83 / 17640 (0.47%)

--- MANAGER_SURVEY ---
Aucune valeur manquante
Total NaN: 0 / 13230 (0.00%)

--- IN_TIME ---


,missing_count,missing_pct
2015-01-01,4410,100.00
2015-11-10,4410,100.00
2015-03-05,4410,100.00
2015-11-11,4410,100.00
2015-11-09,4410,100.00
...,...,...
2015-08-10,197,4.47
2015-04-22,197,4.47
2015-08-18,194,4.40
2015-03-13,191,4.33


Total NaN: 109080 / 1155420 (9.44%)



--- OUT_TIME ---


,missing_count,missing_pct
2015-01-01,4410,100.00
2015-11-10,4410,100.00
2015-03-05,4410,100.00
2015-11-11,4410,100.00
2015-11-09,4410,100.00
...,...,...
2015-08-10,197,4.47
2015-04-22,197,4.47
2015-08-18,194,4.40
2015-03-13,191,4.33


Total NaN: 109080 / 1155420 (9.44%)


##### SECTION – COHERENCE CHECK

Cette cellule vérifie que les identifiants employés sont cohérents entre les différentes sources.

In [5]:
# Contrôle de cohérence des identifiants entre les différentes sources.
ids_general = set(general['EmployeeID'])
ids_employee = set(employee_survey['EmployeeID'])
ids_manager = set(manager_survey['EmployeeID'])
ids_in = set(in_time['EmployeeID'])
ids_out = set(out_time['EmployeeID'])

print('EmployeeID uniques :')
print(f'  general:         {len(ids_general)}')
print(f'  employee_survey: {len(ids_employee)}')
print(f'  manager_survey:  {len(ids_manager)}')
print(f'  in_time:         {len(ids_in)}')
print(f'  out_time:        {len(ids_out)}')

common_ids = ids_general & ids_employee & ids_manager & ids_in & ids_out
print(f'\nIDs communs a tous les fichiers: {len(common_ids)}')

all_sets = {'general': ids_general, 'employee_survey': ids_employee,
            'manager_survey': ids_manager, 'in_time': ids_in, 'out_time': ids_out}

for name_a, set_a in all_sets.items():
    for name_b, set_b in all_sets.items():
        if name_a < name_b:
            diff_ab = set_a - set_b
            diff_ba = set_b - set_a
            if diff_ab or diff_ba:
                print(f'  DIFF {name_a} vs {name_b}: {len(diff_ab)} dans A pas dans B, {len(diff_ba)} dans B pas dans A')

if len(common_ids) == len(ids_general) == len(ids_employee) == len(ids_manager) == len(ids_in) == len(ids_out):
    print('\nCoherence parfaite : tous les EmployeeID sont identiques entre les 5 fichiers.')
else:
    print('\nIncoherence detectee entre les fichiers.')


EmployeeID uniques :
  general:         4410
  employee_survey: 4410
  manager_survey:  4410
  in_time:         4410
  out_time:        4410

IDs communs a tous les fichiers: 4410

Coherence parfaite : tous les EmployeeID sont identiques entre les 5 fichiers.


##### SECTION – EXPORT

On génére ici le fichier de synthèse EDA (`eda_summary.csv`) pour documenter les constats initiaux.


In [6]:
# Export d'un résumé EDA exploitable dans les étapes suivantes.
os.makedirs(PROCESSED_DIR, exist_ok=True)

summary_general = general.describe(include='all').T
summary_general['source'] = 'general'

summary_employee = employee_survey.describe(include='all').T
summary_employee['source'] = 'employee_survey'

summary_manager = manager_survey.describe(include='all').T
summary_manager['source'] = 'manager_survey'

eda_summary = pd.concat([summary_general, summary_employee, summary_manager])
eda_summary.index.name = 'feature'

output_path = os.path.join(PROCESSED_DIR, 'eda_summary.csv')
eda_summary.to_csv(output_path)

print(f'Export OK : {output_path}')
print(f'Shape: {eda_summary.shape}')
display(eda_summary.head(10))

Export OK : ..\data\processed\eda_summary.csv


Shape: (31, 12)


,count,unique,top,freq,mean,std,min,25%,50%,75%,max,source
feature,,,,,,,,,,,,
Age,4410.0,NaN,NaN,NaN,36.92381,9.133301,18.0,30.0,36.0,43.0,60.0,general
Attrition,4410,2,No,3699,NaN,NaN,NaN,NaN,NaN,NaN,NaN,general
BusinessTravel,4410,3,Travel_Rarely,3129,NaN,NaN,NaN,NaN,NaN,NaN,NaN,general
Department,4410,3,Research & Development,2883,NaN,NaN,NaN,NaN,NaN,NaN,NaN,general
DistanceFromHome,4410.0,NaN,NaN,NaN,9.192517,8.105026,1.0,2.0,7.0,14.0,29.0,general
Education,4410.0,NaN,NaN,NaN,2.912925,1.023933,1.0,2.0,3.0,4.0,5.0,general
EducationField,4410,6,Life Sciences,1818,NaN,NaN,NaN,NaN,NaN,NaN,NaN,general
EmployeeCount,4410.0,NaN,NaN,NaN,1.0,0.0,1.0,1.0,1.0,1.0,1.0,general
EmployeeID,4410.0,NaN,NaN,NaN,2205.5,1273.201673,1.0,1103.25,2205.5,3307.75,4410.0,general
